## Token-Level Confidence with `logprobs`

This feature lets you see the model’s **“confidence” score for every token** it generates.  
It’s especially useful for:

- building reliable **classifiers**
- **detecting hallucinations**
- measuring **uncertainty**

### How to Enable It

Include these parameters in your API request payload:

- **`logprobs`**: set to `true`
- **`top_logprobs` (optional)**: an integer **0–20** specifying how many alternative tokens you want to see at each position


In [4]:
import os
import math
import logging
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from openai import OpenAI, APIError, RateLimitError

# 1. Setup Logging (Standard for production observability)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# 2. Load Environment Variables
load_dotenv()

class LLMService:
    def __init__(self):
        """Initialize the client with the API key from environment variables."""
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")
        
        # The client automatically handles retries (default is 2 retries)
        self.client = OpenAI(api_key=api_key)

    def get_completion_with_logprobs(
        self, 
        user_prompt: str, 
        model: str = "gpt-4o", 
        top_logprobs: int = 2
    ) -> Optional[Dict[str, Any]]:
        """
        Fetches completion and processes logprobs into readable probabilities.
        """
        try:
            logger.info(f"Sending request to {model}...")
            
            response = self.client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": user_prompt}
                ],
                logprobs=True,
                top_logprobs=top_logprobs,
                temperature=0.7
            )

            # Extract the content and the logprobs data
            content = response.choices[0].message.content
            logprobs_data = response.choices[0].logprobs.content
            
            # Process logprobs into a structured format
            analyzed_tokens = []
            
            for item in logprobs_data:
                token_str = item.token
                # Convert logprob (natural log) to percentage: e^x * 100
                linear_prob = math.exp(item.logprob) * 100
                
                token_info = {
                    "token": token_str,
                    "confidence_percent": round(linear_prob, 2),
                    "raw_logprob": item.logprob,
                    "top_alternatives": []
                }

                # Process alternatives if requested
                if item.top_logprobs:
                    for alt in item.top_logprobs:
                        alt_prob = math.exp(alt.logprob) * 100
                        token_info["top_alternatives"].append({
                            "token": alt.token,
                            "probability": round(alt_prob, 2)
                        })
                
                analyzed_tokens.append(token_info)

            return {
                "full_text": content,
                "token_analysis": analyzed_tokens
            }

        except RateLimitError:
            logger.error("Rate limit exceeded. Please check your quota.")
            return None
        except APIError as e:
            logger.error(f"OpenAI API returned an API Error: {e}")
            return None
        except Exception as e:
            logger.error(f"An unexpected error occurred: {e}")
            return None

# --- Usage Example ---
if __name__ == "__main__":
    service = LLMService()
    
    # Example Prompt
    prompt = "What is the capital of France?"
    
    result = service.get_completion_with_logprobs(prompt)
    
    if result:
        print(f"\nAnswer: {result['full_text']}\n")
        print("-" * 50)
        print(f"{'Token':<15} | {'Confidence':<12} | {'Alternatives (Top 1)'}")
        print("-" * 50)
        
        for t in result['token_analysis']:
            # Get the top alternative for display (if exists)
            alt_str = ""
            if t['top_alternatives']:
                best_alt = t['top_alternatives'][0]
                alt_str = f"{best_alt['token']} ({best_alt['probability']}%)"
                
            print(f"{repr(t['token']):<15} | {t['confidence_percent']:>5.1f}%       | {alt_str}")

2026-01-05 15:48:52,956 - INFO - Sending request to gpt-4o...
2026-01-05 15:48:54,462 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Answer: The capital of France is Paris.

--------------------------------------------------
Token           | Confidence   | Alternatives (Top 1)
--------------------------------------------------
'The'           | 100.0%       | The (100.0%)
' capital'      | 100.0%       |  capital (100.0%)
' of'           | 100.0%       |  of (100.0%)
' France'       | 100.0%       |  France (100.0%)
' is'           | 100.0%       |  is (100.0%)
' Paris'        | 100.0%       |  Paris (100.0%)
'.'             | 100.0%       | . (100.0%)
